##### The goal of this exercise is to get into some first contact with Python, Pandas data frames, and the many opportunities how to quickly generate static charts from data frames.

For the first exercise, load the provided CSV file, do some data cleaning, and check if your cleaning operation was successful using visualization. For inspiration, browse through some Kaggle notebooks. The original data was downloaded from Kaggle. 

The main purpose of the notebook is to preprocess the CSV file for further visualization. The following steps have to be performed: 

- Read the CSV using Pandas. The data has been slightly preprocessed in Excel to clean and simplify it. See the pandas.read_csv documentation to check how to parse the CSV correctly! (0.5 points)
- Remove unusable rows. The provided CSV may contain rows containing aggregated values or headers we do not need for our final visualization (we are only interested in the individual indicators, not their grouping into pre-defined categories). These rows need to be removed. (0.75 points)
- Take care of missing values. Sci-kit learn provides different data imputation methods. (0.75 points)
    (Hint: You may need to impute before you completely filter your data-frame. You need to make an educated judgment). 
- Visualize the data (twice). Every submitted notebook should contain at least two visualizations using at least two different Python visualization libraries. A list of the most wide-spread Python visualization libraries can be found in this article. You must concisely describe and explain each visualization and your decisions in a Markdown field. You will not receive the points for this task if you did not add a description/explanation. (max. 1.25 points per visualization) Only the first two visualizations will be graded. Submitting more than two visualizations will not result in additional points.
- Save the resulting table as CSV. To be sure that the data is correctly saved, you can load it again. You will have to work with this clean dataset for the second exercise. (0.5 points)


In [7]:
# Setup

import pandas as pd


In [8]:
# Load the dataset / Change later
df = pd.read_csv("./dataset/agriRuralDevelopment.csv")
print(f"Dataset shape: {df.shape[0]:,} observations, {df.shape[1]} features")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print()
print("Column types:")
print(df.dtypes.to_string())

Dataset shape: 13,908 observations, 55 features
Memory usage: 7.10 MB

Column types:
Country Name                                                                                      str
Country Code                                                                                      str
year                                                                                            int64
Access to electricity (% of population)                                                       float64
Access to electricity, rural (% of rural population)                                          float64
Agricultural irrigated land (% of total agricultural land)                                    float64
Agricultural land (% of land area)                                                            float64
Agricultural land (sq. km)                                                                    float64
Agricultural machinery, tractors                                                              float

In [9]:
aggregates = {
    "Arab World",
    "Caribbean small states",
    "Central Europe and the Baltics",
    "East Asia & Pacific",
    "Europe & Central Asia",
    "European Union",
    "Middle East & North Africa",
    "North America",
    "Small states",
    "World"
}

df = df[~df["Country Name"].isin(aggregates)]

In [10]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print(missing_df.to_string())
print(f"\nTotal missing values in entire dataset: {missing.sum()}")

                                                                                            Missing Count  Missing %
Rural land area where elevation is below 5 meters (sq. km)                                          12758      95.94
Rural land area where elevation is below 5 meters (% of total land area)                            12758      95.94
Rural land area (sq. km)                                                                            12758      95.94
Rural population living in areas where elevation is below 5 meters (% of total population)          12758      95.94
Annual freshwater withdrawals, agriculture (% of total freshwater withdrawal)                       12737      95.78
Agricultural irrigated land (% of total agricultural land)                                          12464      93.73
Literacy rate, adult total (% of people ages 15 and above)                                          12250      92.12
Fertilizer consumption (% of fertilizer production)             

In [ ]:
from sklearn.impute import SimpleImputer

id_cols = ["Country Name", "Country Code", "year"]
value_cols = [c for c in df.columns if c not in id_cols]

df[value_cols] = df.groupby("Country Code", group_keys=False)[value_cols].transform(
    lambda col: col.fillna(col.median())
)

usable_value_cols = [c for c in value_cols if not df[c].isna().all()]
all_nan_cols = [c for c in value_cols if df[c].isna().all()]
print(all_nan_cols)

if usable_value_cols:
    imputer = SimpleImputer(strategy="median")
    df[usable_value_cols] = imputer.fit_transform(df[usable_value_cols])

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print(missing_df.to_string())
print(f"\nTotal missing values in entire dataset: {missing.sum()}")

[]
                                                                                            Missing Count  Missing %
Country Name                                                                                            0        0.0
Country Code                                                                                            0        0.0
year                                                                                                    0        0.0
Access to electricity (% of population)                                                                 0        0.0
Access to electricity, rural (% of rural population)                                                    0        0.0
Agricultural irrigated land (% of total agricultural land)                                              0        0.0
Agricultural land (% of land area)                                                                      0        0.0
Agricultural land (sq. km)                                   